<div style="border-left:14px solid #FF7F50;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#FF7F50;letter-spacing:.08em">EUROPEANA</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">What is inside Europeana?</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Distant reading of a collection: who contributes, what is in it, how to ask the API.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 1 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

# Europeana

The IIIF notebook showed what IIIF does *within one platform*. But there are thousands of libraries, archives and museums, each with its own platform. How does a researcher find objects across all of them?

The usual answer is an **aggregator**: a service that collects descriptions of objects from many institutions into one searchable index. [Europeana](https://www.europeana.eu/) is the largest for European cultural heritage.

In [ ]:
# Setup: run this cell first.
from europeana_helpers import *
from iiif_helpers import url_panel, show_mirador

print(API_KEY_STATUS)
overview, institutions, aggregators, countries = load_overview()   # cached counts; fetched once if missing
total_items = overview["total_items"]

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 2 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## How data gets to Europeana

![Europeana flow: institutions reach Europeana directly, through aggregators, or through projects](../misc/assets/img/europeana_flow.svg)

- Institutions run their own **platforms** and offer **metadata** (for harvesting) and, sometimes, **IIIF** (for images).
- Europeana **harvests the metadata** into its own format and reaches institutions *directly*, *via an aggregator* or *via a project*.
- Europeana is mainly an **index of descriptions**. The full-resolution images stay with the institutions.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 3 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## The Europeana API

An **API** lets a program send a request and receive structured data (JSON) instead of a web page. We use the **Search API**, which can return items, or only *counts* per value of a field.

| API | Purpose |
|---|---|
| **Search API** | Query and filter the whole index; counts per value ("facets") |
| **Record API** | Full metadata of one item |

The whole index, counted by media type (`rows=0`: no items, only counts):

In [ ]:
print(f"Items in the index: {total_items:,}\n")
for t, n in facet_counts("TYPE"):
    print(f"  {t:<6} {n:>12,}   {n / total_items:6.1%}")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 4 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Who contributes?

Counting items per institution (`DATA_PROVIDER`). The bar shows each institution's mix of media types.

In [ ]:
show_table(institutions, sort_by="total", n=12, art=True, iiif=True)

Contributions are very uneven. The counts describe what has been *contributed to Europeana*, not the size of an institution's or country's heritage.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 5 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Routes and countries

The `PROVIDER` facet shows the route by which items arrived: national aggregators such as the German Digital Library (DDB), or projects.

In [ ]:
show_table(aggregators, sort_by="total", n=8, first_col="Provider / aggregator")
show_table(countries, sort_by="total", n=8, first_col="Country")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 6 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## A closer look at one institution

The table says how much an institution holds, not what. A few more facet requests restricted to one institution (`qf=DATA_PROVIDER:"name"`) give a quick profile.

In [ ]:
institution_preview("Wellcome Collection")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 7 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Beyond the website's search box

The website filters well. The API is more useful for *numbers across the whole index* and *stacked filters*, including ones the website does not offer, such as a range on the year, or "records that name a IIIF manifest". Each step narrows the previous one:

In [ ]:
steps = [
    ("query: ship",                                   dict(query="ship")),
    ("+ images only",                                 dict(query="ship", qf=["TYPE:IMAGE"])),
    ("+ openly reusable",                             dict(query="ship", qf=["TYPE:IMAGE"], reusability="open")),
    ("+ dated 1800 to 1899",                          dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]"], reusability="open")),
    ("+ record names a IIIF manifest (lower bound)", dict(query="ship", qf=["TYPE:IMAGE", "YEAR:[1800 TO 1899]", "wr_dcterms_isReferencedBy:*"], reusability="open")),
]
for label, kw in steps:
    print(f"{label:<44} {count(**kw):>10,}")

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>EUROPEANA · 8 / 8</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## What one record contains

| Field | Meaning |
|---|---|
| `dataProvider` | the institution that holds the object |
| `provider` | the aggregator or route |
| `edmIsShownAt` | the object's page on the institution's website |
| `edmIsShownBy` | the media file on the institution's server |
| `edmPreview` | a small preview image served by Europeana |
| `rights` | the rights statement |

In [ ]:
sample_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE"], theme="art", reusability="open")
sample = sample_results["items"]
item = sample[0]
for field in ["title", "dataProvider", "provider", "country", "type", "rights", "edmIsShownAt", "edmIsShownBy", "edmPreview"]:
    value = item.get(field)
    if isinstance(value, list):
        value = value[0]
    print(f"{field:<14} {value}")

<div style="margin-top:40vh;border-left:14px solid #C0C0C0;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#C0C0C0;letter-spacing:.08em">APPENDIX</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Background</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Details for reading afterwards. Not part of the presentation.</div></div>

## Europeana · A. How the data gets to Europeana in more detail

From the bottom up (see the diagram in the Europeana section):

- Institutions run their own **platforms** (for example e-codices or e-rara for Swiss manuscripts and rare prints). A platform exposes its data in two separate ways:
  - **Metadata** (title, creator, date, rights, …) through a harvesting protocol, typically **OAI-PMH**, in formats such as METS/MODS or Dublin Core.
  - **Images and manifests** through **IIIF**, as in the IIIF notebook. Viewers and research tools read these directly.
- Europeana **harvests the metadata** and converts it to its own format, the Europeana Data Model (**EDM**). Institutions reach Europeana by one of three routes:
  - *directly*, for large providers;
  - *via an aggregator*, a national or domain-specific one such as Germany's Deutsche Digitale Bibliothek (DDB);
  - *via a project*, for thematic EU-funded projects.
- Where an institution also offers IIIF, the Europeana record can link to its manifest (the dashed line).

Two consequences are worth keeping in mind:

- Europeana is mainly an **index of descriptions**. A record holds metadata, a small preview image, and links back to the institution's own page and media file. The full-resolution objects stay with the institutions.
- Europeana records carry two names: the **data provider** (the institution that holds the object) and the **provider** (the aggregator or route that delivered it). The API lets you count by either, and the tables below use both.

## Europeana · B. The Search API in more detail

Requests carry a free API key (`wskey`). Create your own at <https://pro.europeana.eu/page/get-api> and save it to `notebooks/api-key-europeana.txt`. Without one the notebook uses a shared demo key, which is enough for everything here but is rate limited.

| Parameter | Meaning | Example |
|---|---|---|
| `query` | search text (`*` for everything) | `Rembrandt` |
| `qf` | filter on a field | `TYPE:IMAGE`, `COUNTRY:Sweden` |
| `theme` | a thematic collection | `art`, `manuscripts`, `photography` |
| `rows` | number of results (max 100) | `12` |
| `profile` | how much to return (`standard`, `rich`, `facets`) | `facets` |
| `facet` | count items per value of a field | `DATA_PROVIDER` |

The Record API returns the full metadata of one item: `…/record/v2/{dataset}/{local_id}.json`. The helper functions are in `europeana_helpers.py`.

## Europeana · C. Counting institutions in more detail

The Search API can count items per value of a field. Four fields are relevant here:

| Facet | Meaning |
|---|---|
| `DATA_PROVIDER` | the institution that holds the object |
| `PROVIDER` | the aggregator or route through which it reached Europeana |
| `COUNTRY` | the country of the data provider |
| `TYPE` | the media type of the item: `IMAGE`, `TEXT`, `SOUND`, `VIDEO`, `3D` |

Europeana's `TYPE` values describe the media, not the genre. In practice `TEXT` is where books, manuscripts, newspapers and archival documents sit, and `IMAGE` holds paintings, photographs, maps and prints. "Art" is not a type but a **thematic collection** (`theme=art`) that cuts across types, so it is counted separately and overlaps with the other columns.

To split every institution by type, `load_overview()` asks for the `DATA_PROVIDER` counts once per type (with `qf=TYPE:…`), and once more for the Art theme. The same is done for aggregators (`PROVIDER`) and countries. That is about 35 small requests in total. A further column counts the records that link to a IIIF manifest (`wr_dcterms_isReferencedBy:*`). The results are saved to `data/europeana/overview_cache.json`, so later runs read the file instead of calling the API. Call `load_overview(refresh=True)` to fetch fresh numbers.

Because `TYPE`, the Art theme and the IIIF column are counted separately, the type columns add up to the total (an item has exactly one type), while the Art and IIIF-linked columns are subsets that overlap them.

Which institutions contribute **3D objects**?

In [ ]:
show_table(institutions, sort_by="3D", n=15, art=True, iiif=True)

And which hold the most **text** (books, manuscripts, newspapers)?

In [ ]:
show_table(institutions, sort_by="TEXT", n=15, art=True, iiif=True)

How uneven are the contributions?

In [ ]:
totals = sorted((r["total"] for r in institutions), reverse=True)
def share(k): return sum(totals[:k]) / sum(totals)
print(f"Institutions with at least 1 item:        {len(totals):,}")
print(f"Median institution:                       {totals[len(totals)//2]:,} items")
print(f"Top 10 institutions hold                  {share(10):.0%} of all items")
print(f"Top 100 institutions hold                 {share(100):.0%} of all items")
print(f"Institutions with fewer than 1,000 items: {sum(t < 1000 for t in totals):,}")

To keep the numbers for slides or further analysis, the tables can be written to CSV:

In [ ]:
for rows, name in [(institutions, "overview_institutions.csv"), (aggregators, "overview_providers.csv"), (countries, "overview_countries.csv")]:
    print("wrote", write_csv(rows, name).relative_to(PROJECT_ROOT))

## Europeana · D. Another institution

The samples in the profile are taken from a random position within the first 1,000 results (the API does not page deeper with `start`), so they come from the front of the institution's result list in Europeana's default order, not from a true random draw. Name the institution exactly as it appears in the tables:

In [ ]:
INSTITUTION = "Wellcome Collection"      # <-- change this, e.g. "Deutsche Fotothek", "Rijksmuseum", "Naturalis Biodiversity Center"

institution_preview(INSTITUTION)

## Europeana · E. Beyond the search box in more detail

### Distributions

A facet request returns counts per value of a field for any query. The website shows facets as a sidebar for the current search. Through the API they can be requested for any field and collected into a table or a chart. For example, for the query `river`:

In [ ]:
QUERY = "river"
total_river, by_institution = facet_pairs("DATA_PROVIDER", QUERY, limit=8)
_, years = facet_pairs("YEAR", QUERY, limit=2000)

by_century = {}
for y, c in years:
    if y.isdigit() and 1000 <= int(y) <= 2025:
        by_century[int(y) // 100 + 1] = by_century.get(int(y) // 100 + 1, 0) + c

display(HTML(
    f"<h4 style='font-family:sans-serif'>'{QUERY}': {total_river:,} items</h4><div style='display:flex;gap:40px;flex-wrap:wrap'>"
    + bar_table(by_institution, by_institution[0][1], "Institutions with the most matches")
    + bar_table(sorted(by_century.items()), max(by_century.values()), "Dated items by century (11th to 21st)", century_label)
    + "</div>"))

A cross-tabulation combines two fields, for example rights statements per media type. It takes one request per row:

In [ ]:
table = {}
for t in ["IMAGE", "TEXT", "3D"]:
    n, pairs = facet_pairs("RIGHTS", "*", limit=6, qf=f"TYPE:{t}")
    table[t] = [(short_rights(k), v / n) for k, v in pairs]

display(HTML("<div style='display:flex;gap:40px;flex-wrap:wrap;font-family:sans-serif'>" + "".join(
    "<div><b>" + t + "</b><table style='font:12px sans-serif'>"
    + "".join(f"<tr><td style='padding-right:10px'>{html.escape(k)}</td><td style='text-align:right'>{v:.0%}</td></tr>" for k, v in rows)
    + "</table></div>" for t, rows in table.items()) + "</div>"))

### Export metadata as a dataset

The Search API pages through results with a *cursor*, so any number of records can be collected, not only the first 1,000. The cell below collects a few hundred records for a query into a table and saves it as CSV. Counting and sorting that table is then ordinary data analysis, outside any website. The same approach, scaled up, is how the shared dataset for the next notebook is built.

Results arrive in Europeana's default order, so a single institution can dominate (look at the counts printed below). A balanced dataset needs one query per institution.

In [ ]:
records = export_metadata("ship", qf=["TYPE:IMAGE", "wr_dcterms_isReferencedBy:*"])
out = EUROPEANA_DIR / "export_ship.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(records[0]))
    w.writeheader(); w.writerows(records)

print(f"{len(records)} records saved to {out.relative_to(PROJECT_ROOT)}\n")
print("Institutions:", Counter(r["institution"] for r in records).most_common(5))
print("Countries:   ", Counter(r["country"] for r in records).most_common(5))
print("Creators:    ", Counter(r["creator"] for r in records).most_common(5))

### What is not available through the API

The API returns metadata and links, not the objects. The full-resolution images stay on the institution's server, which is where the IIIF technique from the IIIF notebook comes in: with the `edmIsShownBy` link (or the manifest), a program can request a small version, a region, or a deep-zoom tile of each item. The metadata export above is therefore the first half of building an image dataset, and the IIIF image request is the second.

## Europeana · F. Which institutions run their own IIIF?

Europeana builds a manifest for every record, whether or not the institution serves IIIF itself. The table counts something narrower: records where the institution names its own IIIF resource (`wr_dcterms_isReferencedBy`). For these, an image service is likely, so the techniques from the IIIF notebook (small versions, regions, deep zoom) apply directly. The count is a **lower bound**: an institution can serve IIIF without naming it in its Europeana records.


In [ ]:
iiif_top = sorted((r for r in institutions if r["iiif"]), key=lambda r: r["iiif"], reverse=True)
share = sum(r["iiif"] for r in institutions) / total_items
print(f"{len(iiif_top):,} of {len(institutions):,} institutions name their own IIIF resource in at least one record ({share:.1%} of all records)\n")
show_table(institutions, sort_by="iiif", n=10, iiif=True)

## Europeana · G. From a record to a IIIF manifest

Europeana publishes a manifest for each record at a predictable address, built from the record ID. That connects this notebook to the IIIF notebook: the manifest lists the canvases of the object, and its `service` entries, if present, are IIIF image services at the institution.

In [ ]:
print("manifest of the first record:")
print(get_iiif_manifest_url(item))

flags = check_manifests(sample)
for it, flag in zip(sample, flags):
    mark = {True: "image service", False: "static image only", None: "no manifest"}[flag]
    print(f"{mark:<18} {get_item_title(it)[:50]:<52} {first_of(it.get('dataProvider'))[:35]}")
print(f"\n{sum(f is True for f in flags)} of {len(sample)} records have an IIIF image service.")

Not every record has an image service. Where a manifest with one exists, the techniques from the IIIF notebook apply: region requests, deep zoom and annotations. Where an institution only supplies a plain image file, the record is still searchable, but its pixels cannot be requested in parts. The manifest check on each record is the reliable test.

## Europeana · H. The IIIF filter and the manifest check

The same manifest check on records selected with the IIIF filter (`wr_dcterms_isReferencedBy:*`):

In [ ]:
iiif_results = search_europeana(query="portrait", rows=20, qf=["TYPE:IMAGE", "wr_dcterms_isReferencedBy:*"], theme="art", reusability="open")
iiif_sample = iiif_results["items"]
iiif_flags = check_manifests(iiif_sample)
print(f"{sum(f is True for f in iiif_flags)} of {len(iiif_sample)} records selected with the IIIF filter have an image service.")

The IIIF filter is only reliable in one direction: records selected with it do have an image service, but records without the filter can have one too (the Wellcome Collection is an example), so the manifest check on each record is the reliable test.

## Europeana · I. Your turn: find an object with a IIIF manifest

*About 10 minutes. Edit the cells and run them again (Shift+Enter).*

Search Europeana for something that interests you, narrow it down with filters, and pick one result.

| Setting | Meaning | Examples |
|---|---|---|
| `SEARCH_KEYWORD` | what to search for | `"portrait"`, `"map"`, `"horse"`, `"cathedral"` |
| `YEARS` | period, or `None` | `(1800, 1899)` |
| `ONLY_IIIF` | only records that name a IIIF manifest | `True` / `False` |
| `THEME` | a thematic collection, or `None` | `"art"`, `"photography"`, `"manuscripts"` |

In [ ]:
# ============================================================
# EXERCISE: combine a keyword with filters
# ============================================================
SEARCH_KEYWORD = "ship"        # <-- try "portrait", "map", "horse", "cathedral"
YEARS          = (1800, 1899)  # <-- or None for any period
ONLY_IIIF      = True          # <-- True: only records that name a IIIF manifest
THEME          = None          # <-- or "art", "manuscripts", "photography", ...
# ============================================================

qf = ["TYPE:IMAGE"]
if YEARS:
    qf.append(f"YEAR:[{YEARS[0]} TO {YEARS[1]}]")
if ONLY_IIIF:
    qf.append("wr_dcterms_isReferencedBy:*")

results = search_europeana(query=SEARCH_KEYWORD, rows=12, qf=qf, theme=THEME, reusability="open")
if results and results.get("success"):
    items = results["items"]
    print(f"{results['totalResults']:,} matching images (showing {len(items)})\n")
    show_results(items)
else:
    print("Search failed")

### Tasks

1. Change the **keyword** to something you care about and run again.
2. Change the **period**, or set `YEARS = None`.
3. Set `ONLY_IIIF = False` and compare the total with `True`. How many records name a manifest?
4. Choose one result you like. Note its number (`#0`, `#1`, ...).

## Europeana · J. Exercise: from a result to its manifest

Enter the number of your pick. The cell builds the IIIF manifest URL for the record and checks whether its images are served by a IIIF image service.

In [ ]:
PICK = 0        # <-- the number of the result you chose

chosen = items[PICK]
manifest_url = get_iiif_manifest_url(chosen)
has_service = manifest_has_image_service(chosen)

print(get_item_title(chosen))
print(first_of(chosen.get("dataProvider")), "|", chosen["id"])
print("IIIF image service:", {True: "yes", False: "no, a static image only", None: "no manifest found"}[has_service])

url_panel("Your manifest", [("IIIF manifest URL", manifest_url, "Copy it. You can paste it into the Mirador demo from the IIIF notebook, Add resource.")])

In [ ]:
show_mirador(manifest_url)

### Stretch

- Go to the **Mirador demo** from the IIIF notebook and add your manifest there. Open a second window with the manifest of a neighbour.
- Profile the institution of your pick: `institution_preview("name")`, with the name from the cell above.
- Look at the record in the API: print `chosen["rights"]` and `chosen["edmIsShownAt"]`. What may you do with the image?

<details><summary><b>Hints</b> (click to open)</summary>

- No results? Remove one filter at a time, starting with `YEARS` and `ONLY_IIIF`.
- A result without an image service still has a manifest, but it shows a static image only. Pick another result if you want to zoom.
- `institution_preview("Wellcome Collection")` shows the exact spelling the API expects. Names are in the tables of the Europeana section.
</details>

## Europeana · K. Summary

- Europeana is an **aggregator**: it harvests metadata (as EDM) from institutions, directly, through national or domain aggregators, or through projects. The full-resolution media stay with the institutions.
- IIIF and aggregation are complementary. Aggregation helps with *finding* objects across institutions, and IIIF helps with *viewing and using* them once found.
- The Search API can return items or counts. Counting by `DATA_PROVIDER`, `PROVIDER`, `COUNTRY` and `TYPE` gives an overview of who contributes what.
- The API is most useful for what a search box does not do: distributions over a field, stacked and range filters, and exporting metadata as a table.
- Contributions are very uneven, and the `TYPE` field describes media, not genre.

### Resources

- Europeana APIs: <https://pro.europeana.eu/page/apis>
- Search API reference: <https://pro.europeana.eu/page/search>
- Europeana Data Model (EDM): <https://pro.europeana.eu/page/edm-documentation>

## Next

The next notebook works with a shared dataset: small images from many of these institutions, collected through exactly the requests shown here and through IIIF, and their precomputed CLIP embeddings. You will search that collection with your own text queries.